## TP 10 Détection des mentions de personnages avec BERT (ANTOINE)
Usage de perceptron mono-couche pour prédire si l'embeddings de BERT correspond à une mention de personnage. S'il y a le temps : faire un perceptron multicouche pour faire la même tâche.

In [1]:
from datasets import load_dataset

dataset = load_dataset("lattice-nlp/litbank-fr-PER")["train"].to_list()

print(dataset[0].keys())

dict_keys(['file_name', 'language', 'text', 'stanza_tokens', 'mentions', 'sentence_spans'])


### Exploration Structure Dataset

In [2]:
print(f"{len(dataset):,} documents\n")

example_document = dataset[0]

for key, value in example_document.items():
    print(f"{key:<20} {type(value)}    {repr(value)[:100]}")

32 documents

file_name            <class 'str'>    '1882-1883_Maupassant-Guy-de_Mademoiselle-Fifi-La-buche'
language             <class 'str'>    'fr'
text                 <class 'str'>    "LA BUCHE\n\nLe salon était petit, tout enveloppé de tentures épaisses, et discrètement odorant. Dan
stanza_tokens        <class 'int'>    2249
mentions             <class 'list'>    [{'onset': 278, 'offset': 296, 'COREF': 'deux_personnes', 'type': 'PER'}, {'onset': 297, 'offset': 3
sentence_spans       <class 'list'>    [[0, 10], [10, 94], [94, 313], [313, 762], [762, 879], [879, 905], [905, 1137], [1137, 1217], [1217,


In [3]:
print(f"{len(example_document["mentions"]):,} mentions\n")

example_mention = example_document["mentions"][0]
for key, value in example_mention.items():
    print(f"{key:<20} {type(value)}    {repr(value)[:100]}")

235 mentions

onset                <class 'int'>    278
offset               <class 'int'>    296
COREF                <class 'str'>    'deux_personnes'
type                 <class 'str'>    'PER'


### Ajouter le champ "text" pour chaque mention. Récupérer l'empans correspondant à `onset`:`offset`.

In [4]:
for document in dataset:
    for mention in document["mentions"]:
        mention["text"] = document["text"][mention["onset"]:mention["offset"]]

In [5]:
dataset[0]["mentions"][:10]

[{'onset': 278,
  'offset': 296,
  'COREF': 'deux_personnes',
  'type': 'PER',
  'text': 'les deux personnes'},
 {'onset': 297,
  'offset': 300,
  'COREF': 'deux_personnes',
  'type': 'PER',
  'text': 'qui'},
 {'onset': 313,
  'offset': 317,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'Elle'},
 {'onset': 320,
  'offset': 345,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'la maîtresse de la maison'},
 {'onset': 347,
  'offset': 375,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'une vieille à cheveux blancs'},
 {'onset': 382,
  'offset': 411,
  'COREF': 'vieille',
  'type': 'PER',
  'text': 'une de ces vieilles adorables'},
 {'onset': 684, 'offset': 688, 'COREF': 'M1', 'type': 'PER', 'text': 'vous'},
 {'onset': 762, 'offset': 765, 'COREF': 'paul', 'type': 'PER', 'text': 'Lui'},
 {'onset': 807,
  'offset': 836,
  'COREF': 'paul',
  'type': 'PER',
  'text': 'un ami de toutes les semaines'},
 {'onset': 838,
  'offset': 877,
  'COREF': 'paul',
  'type': 'PER',
  'text': "un comp

### Quel est le texte le plus long dans le dataset ?

In [12]:
document_len_dict = {document["file_name"]:len(document["text"]) for document in dataset}
document_len_dict = dict(
    sorted(document_len_dict.items(), key=lambda x: x[1], reverse=True)
)
longest_doc_file_name = list(document_len_dict.keys())[0]
print(longest_doc_file_name)

1832_Sand-George_Indiana


### Qui en sont les 5 personnages principaux

Voir : [https://fr.wikipedia.org/wiki/Indiana_(roman)#Personnages](https://fr.wikipedia.org/wiki/Indiana_(roman)#Personnages)

In [26]:
document = [document for document in dataset if document["file_name"] in document["file_name"]==longest_doc_file_name][0]

from collections import Counter

Counter([mention["COREF"] for mention in document["mentions"]]).most_common(5)

[('Indiana_Delmare_nee_de_Carvajal', 4929),
 ('Raymon_de_Ramiere', 3627),
 ('sir_Rodolphe_Brown_dit_Ralph', 2265),
 ('colonel_Delmare', 1292),
 ('Noun', 735)]

In [18]:
document["mentions"]

TypeError: list indices must be integers or slices, not str